# WC 2022 Calibration Backtest

Robust full-tournament Monte Carlo to evaluate **trained model calibration**.

**Prerequisite:** `/kaggle/working/models/` from the WC 2026 training notebook (or attach a dataset with `calibration.json`, GBM, NN, Bayesian artifacts).

Attach the same **`soccer-data`** dataset.

Actual champion: **Argentina**

1. **Cell 1** — setup
2. **Cell 2** — quick smoke (`500` sims)
3. **Cell 3** — robust run (`50k` sims, ~1-3h)


In [ ]:
import base64
import io
import json
import os
import subprocess
import sys
import zipfile
from pathlib import Path

os.environ.setdefault("KMP_DUPLICATE_LIB_OK", "TRUE")
os.environ.setdefault("OMP_NUM_THREADS", "1")

REPO_ZIP_B64 = """..."""

PIP_PACKAGES = [
    "pandas>=2.0",
    "pyarrow>=14.0",
    "pyyaml>=6.0",
    "lightgbm>=4.0",
    "numpy>=1.24",
    "scipy>=1.10",
    "tqdm>=4.66",
    "torch>=2.0",
    "pymc>=5.0",
    "arviz>=0.16",
    "matplotlib>=3.7",
]

WORK = Path("/kaggle/working")
REPO = WORK / "WorldCupPredictor"
MODELS = WORK / "models"
DATA_ROOT: Path | None = None

REQUIRED_DATA_FILES = (
    "results.csv",
    "wc2026_results.csv",
    "former_names.csv",
    "fixtures.csv",
    "match_stats.csv",
    "understat_matches.parquet",
)


def install_dependencies() -> None:
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", *PIP_PACKAGES]
    )


def extract_project() -> Path:
    if REPO.exists():
        return REPO
    WORK.mkdir(parents=True, exist_ok=True)
    payload = base64.b64decode(REPO_ZIP_B64)
    with zipfile.ZipFile(io.BytesIO(payload)) as zf:
        zf.extractall(WORK)
    if not REPO.exists():
        raise RuntimeError(f"Expected project at {REPO} after extract")
    subprocess.check_call(
        [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)]
    )
    return REPO


def discover_data_root() -> Path:
    kaggle_input = Path("/kaggle/input")
    if not kaggle_input.is_dir():
        raise FileNotFoundError(
            "/kaggle/input not found. Attach the soccer-data dataset to this notebook."
        )

    def score(root: Path) -> int:
        return sum(1 for name in REQUIRED_DATA_FILES if (root / name).exists())

    candidates: list[Path] = []
    seen: set[str] = set()

    def add(path: Path) -> None:
        key = str(path.resolve())
        if key in seen:
            return
        seen.add(key)
        candidates.append(path)

    for results_csv in kaggle_input.rglob("results.csv"):
        add(results_csv.parent)
    for item in sorted(kaggle_input.iterdir()):
        if item.is_dir():
            add(item)
            for sub in sorted(item.rglob("*")):
                if sub.is_dir():
                    add(sub)

    ranked = sorted(((score(path), path) for path in candidates), key=lambda x: (-x[0], str(x[1])))
    preferred = ("soccer-data", "soccer-data-dataset")
    for found_score, path in ranked:
        if found_score >= 3 and (
            path.name in preferred or any(part in preferred for part in path.parts)
        ):
            return path
    for found_score, path in ranked:
        if found_score >= 3:
            return path

    lines = ["Could not find soccer data files."]
    if not candidates:
        lines.append("/kaggle/input is empty. Click Add Data and attach soccer-data.")
    else:
        lines.append("/kaggle/input contents:")
        for path in candidates:
            files = sorted(p.name for p in path.iterdir() if p.is_file())
            lines.append(f"  {path}: {', '.join(files[:8])}")
    raise FileNotFoundError("\n".join(lines))


def verify_data() -> Path:
    global DATA_ROOT
    DATA_ROOT = discover_data_root()
    missing = [name for name in REQUIRED_DATA_FILES if not (DATA_ROOT / name).exists()]
    if missing:
        raise FileNotFoundError(f"Missing in {DATA_ROOT}: {missing}")
    print("Data OK:", DATA_ROOT)
    return DATA_ROOT


def verify_models() -> Path:
    cal = MODELS / "calibration.json"
    if not cal.exists():
        raise FileNotFoundError(
            f"Missing {cal}. Run the WC 2026 training notebook first, "
            "or copy trained artifacts into /kaggle/working/models/."
        )
    print("Models OK:", MODELS)
    return MODELS

YEAR = 2022
ACTUAL_CHAMPION = "Argentina"
BACKTEST_JSON = MODELS / f"wc{{YEAR}}_calibration_backtest.json"
SCRIPT = REPO / "scripts" / f"run_calibration_backtest_{{YEAR}}.py"


def run_backtest(
    profile: str = "backtest",
    n_sims: int | None = None,
    seed: int = 42,
) -> int:
    """Robust full-tournament MC using trained models in /kaggle/working/models."""
    repo = extract_project()
    data_root = verify_data()
    verify_models()
    cmd = [
        sys.executable,
        str(SCRIPT),
        "--profile", profile,
        "--seed", str(seed),
        "--models-dir", str(MODELS),
        "--data-root", str(data_root),
    ]
    if n_sims is not None:
        cmd.extend(["--sims", str(n_sims)])
    print("Running:", " ".join(cmd))
    result = subprocess.run(cmd, check=False)
    if result.returncode != 0:
        raise RuntimeError(f"Backtest failed with exit code {{result.returncode}}")
    return result.returncode


def show_backtest_results(top_n: int = 5) -> None:
    if not BACKTEST_JSON.exists():
        print(f"No report yet at {{BACKTEST_JSON}}")
        return
    report = json.loads(BACKTEST_JSON.read_text(encoding="utf-8"))
    metrics = report["calibration_metrics"]
    print(f"WC {{YEAR}} calibration backtest — {{report['n_sims']}} simulations")
    print(f"Actual champion: {{metrics['actual_champion']}} ({{ACTUAL_CHAMPION}})")
    print(f"P(actual champion): {{metrics['p_actual_champion']:.4f}}")
    print(f"Champion rank: {{metrics['actual_champion_rank']}}")
    print(f"Multiclass Brier: {{metrics['multiclass_brier']:.4f}}")
    print(f"Log score: {{metrics['log_score']:.4f}}")
    print(f"Elapsed: {{report['elapsed_seconds']}}s")
    print("\nTop champion probabilities:")
    for row in report["top_champion_probabilities"][:top_n]:
        print(f"  {{row['team']}}: {{row['prob']:.4f}}")
    print(f"\nReport: {{BACKTEST_JSON}}")


install_dependencies()
extract_project()
verify_data()
try:
    verify_models()
except FileNotFoundError as exc:
    print("WARNING:", exc)
    print("Run the WC 2026 training notebook first, then re-run this cell.")
else:
    print("Setup complete. Next: run_backtest(profile='fast', n_sims=500)")


In [ ]:
# Quick smoke (~5-20 min). Requires models from WC 2026 notebook.
run_backtest(profile='fast', n_sims=500)
show_backtest_results()


In [ ]:
# Robust calibration backtest (50k sims by default).
run_backtest(profile='backtest')
show_backtest_results(top_n=10)
